# Proyecto 2 – Análisis Exploratorio
## Reto 11 – Negocios: LMSYS Chatbot Arena Human Preference Predictions

**CC3084 Data Science | Universidad del Valle de Guatemala | Semestre II 2026**

| Integrante | Carné |
|---|---|
| Mia Alejandra Fuentes Mérida | 23775 |
| Roberto José Barreda Siekavizza | 23354 |
| Javier Eduardo España Pacheco | 23361 |
| Angel Esteban Esquit Hernández | 23221 |

---

## Parte 2 – Descripción y Limpieza de Datos

**Prerequisito:** Haber ejecutado `01_introduccion_y_carga.ipynb` (o tener `data/raw/train.csv` descargado).


---
## 1. Configuración del Ambiente


In [ ]:
import sys
import json
from pathlib import Path

# Agregar el directorio raíz al path para importar src/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import load

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Ambiente configurado correctamente.')


In [ ]:
# Cargamos el dataset crudo (requiere haberlo descargado en la Parte 1)
df = load.load_train()
print(f'Shape de train.csv: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.head(3)


---
## 2. Descripción de Variables

Tabla con el nombre, tipo, descripción y rango/cardinalidad de cada columna del dataset crudo.


In [ ]:
descripcion_vars = pd.DataFrame([
    {'columna': 'id',              'tipo': 'str/int', 'descripcion': 'Identificador único de la batalla',                              'rango_cardinalidad': f'{df[cfg.COL_ID].nunique():,} valores únicos'},
    {'columna': 'model_a',         'tipo': 'categórica','descripcion': 'Nombre del primer modelo (posición A)',                         'rango_cardinalidad': f'{df[cfg.COL_MODEL_A].nunique()} modelos distintos'},
    {'columna': 'model_b',         'tipo': 'categórica','descripcion': 'Nombre del segundo modelo (posición B)',                        'rango_cardinalidad': f'{df[cfg.COL_MODEL_B].nunique()} modelos distintos'},
    {'columna': 'prompt',          'tipo': 'texto (JSON list)', 'descripcion': 'Turnos de conversación emitidos por el usuario',         'rango_cardinalidad': 'Lista de strings serializada como JSON'},
    {'columna': 'response_a',      'tipo': 'texto (JSON list)', 'descripcion': 'Respuestas del modelo A, un elemento por turno',         'rango_cardinalidad': 'Lista de strings serializada como JSON, puede contener None'},
    {'columna': 'response_b',      'tipo': 'texto (JSON list)', 'descripcion': 'Respuestas del modelo B, un elemento por turno',         'rango_cardinalidad': 'Lista de strings serializada como JSON, puede contener None'},
    {'columna': 'winner_model_a',  'tipo': 'binaria (0/1)', 'descripcion': '1 si el usuario prefirió la respuesta del modelo A',         'rango_cardinalidad': '{0, 1}'},
    {'columna': 'winner_model_b',  'tipo': 'binaria (0/1)', 'descripcion': '1 si el usuario prefirió la respuesta del modelo B',         'rango_cardinalidad': '{0, 1}'},
    {'columna': 'winner_tie',      'tipo': 'binaria (0/1)', 'descripcion': '1 si el usuario declaró un empate entre ambas respuestas',   'rango_cardinalidad': '{0, 1}'},
])
descripcion_vars


Cada fila del dataset representa una **batalla**: un usuario emite uno o más turnos de conversación
(`prompt`), recibe respuestas de dos modelos anónimos (`response_a`, `response_b`) y elige cuál prefiere,
codificado a través de las tres columnas mutuamente excluyentes `winner_model_a`, `winner_model_b` y
`winner_tie` (exactamente una de las tres vale 1 por fila).


---
## 3. Análisis de Valores Nulos


In [ ]:
nulos = df.isnull().sum().rename('nulos')
nulos_pct = (df.isnull().mean() * 100).round(2).rename('% nulos')
resumen_nulos = pd.concat([nulos, nulos_pct], axis=1).sort_values('% nulos', ascending=False)
resumen_nulos


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=resumen_nulos.index, y=resumen_nulos['% nulos'], color='#C44E52', ax=ax)
ax.set_title('Porcentaje de valores nulos por columna')
ax.set_ylabel('% nulos')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../docs/figures/fig_nulos_por_columna.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en docs/figures/fig_nulos_por_columna.png')


**Decisión sobre nulos:** las columnas `response_a` y `response_b` son listas JSON y pueden contener
elementos `None` dentro de la lista (un turno sin respuesta), no necesariamente la celda completa nula.
Esos `None` se manejan al extraer el texto plano en la sección 5 (se reemplazan por cadena vacía, no se
elimina la fila, porque la batalla completa sigue siendo válida). Si alguna columna clave (`id`, `prompt`,
las tres columnas de `winner`) tuviera nulos a nivel de celda, esas filas sí se eliminarían por no poder
derivarse el resultado de la batalla.


---
## 4. Detección y Manejo de Duplicados


In [ ]:
n_duplicados = df.duplicated(subset=cfg.COL_ID).sum()
print(f'Filas con id duplicado: {n_duplicados}')

if n_duplicados > 0:
    df = df.drop_duplicates(subset=cfg.COL_ID, keep='first').reset_index(drop=True)
    print(f'Se eliminaron los duplicados. Nuevo shape: {df.shape[0]:,} filas x {df.shape[1]} columnas')
else:
    print('No se encontraron ids duplicados, no se elimina ninguna fila.')


---
## 5. Limpieza de Columnas de Texto

Las columnas `prompt`, `response_a` y `response_b` vienen serializadas como *strings* JSON que
representan una lista de turnos de conversación (uno por cada intercambio). Se extraen a texto plano,
uniendo los turnos con salto de línea y reemplazando los `None` internos por cadena vacía.


In [ ]:
def parse_turns(json_str):
    """Convierte un string JSON de lista de turnos en una lista de strings (None -> '')."""
    try:
        turns = json.loads(json_str)
    except (json.JSONDecodeError, TypeError):
        return []
    if turns is None:
        return []
    return [t if t is not None else '' for t in turns]


def turns_to_text(json_str):
    """Une los turnos de conversación en un solo texto plano."""
    return '\n'.join(parse_turns(json_str)).strip()


df['prompt_text'] = df[cfg.COL_PROMPT].apply(turns_to_text)
df['resp_a_text'] = df[cfg.COL_RESP_A].apply(turns_to_text)
df['resp_b_text'] = df[cfg.COL_RESP_B].apply(turns_to_text)

df[['prompt_text', 'resp_a_text', 'resp_b_text']].head(3)


---
## 6. Derivar la Columna `winner`

Se unifica el resultado de la batalla en una sola columna categórica, usando `config.LABEL_MAP`:
0 = gana model_a, 1 = gana model_b, 2 = empate.


In [ ]:
df['winner'] = np.select(
    [df[cfg.COL_WIN_A] == 1, df[cfg.COL_WIN_B] == 1, df[cfg.COL_WIN_TIE] == 1],
    [cfg.LABEL_MAP[cfg.COL_WIN_A], cfg.LABEL_MAP[cfg.COL_WIN_B], cfg.LABEL_MAP[cfg.COL_WIN_TIE]],
    default=-1,
).astype(int)

# Verificación: ninguna fila debería quedar sin clasificar (las tres columnas de ganador
# son mutuamente excluyentes y exhaustivas en el dataset original)
filas_sin_clasificar = (df['winner'] == -1).sum()
print(f'Filas sin ganador identificado: {filas_sin_clasificar}')

df['winner'].map(cfg.LABEL_NAMES).value_counts()


---
## 7. Features de Longitud

Se calculan longitudes en caracteres y en palabras del prompt y de cada respuesta, además del número
de turnos de conversación por batalla.


In [ ]:
df['prompt_len'] = df['prompt_text'].str.len()
df['resp_a_len'] = df['resp_a_text'].str.len()
df['resp_b_len'] = df['resp_b_text'].str.len()

df['prompt_words'] = df['prompt_text'].str.split().str.len()
df['resp_a_words'] = df['resp_a_text'].str.split().str.len()
df['resp_b_words'] = df['resp_b_text'].str.split().str.len()

df['n_turns'] = df[cfg.COL_PROMPT].apply(lambda x: len(parse_turns(x)))

cols_longitud = ['prompt_len', 'resp_a_len', 'resp_b_len', 'prompt_words', 'resp_a_words', 'resp_b_words', 'n_turns']
df[cols_longitud].describe()


---
## 8. Guardar Dataset Limpio


In [ ]:
cfg.PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(cfg.F_TRAIN_CLEAN, index=False)
print(f'Dataset limpio guardado en: {cfg.F_TRAIN_CLEAN}')
print(f'Shape final: {df.shape[0]:,} filas x {df.shape[1]} columnas')
df.columns.tolist()


---
## 9. Resumen de la Limpieza

- Se identificaron y documentaron los valores nulos: los `None` dentro de `response_a` / `response_b`
  corresponden a turnos sin respuesta, se manejan a nivel de texto (no se descartan filas por esto).
- Se revisaron duplicados por `id` y se eliminaron si existían.
- Se extrajo el texto plano de `prompt`, `response_a` y `response_b` a partir de las listas JSON.
- Se derivó la columna unificada `winner` (0 = model_a, 1 = model_b, 2 = empate) a partir de las tres
  columnas binarias originales.
- Se calcularon features de longitud (`*_len`, `*_words`) y `n_turns`, que se usarán en la Parte 3
  (EDA numérico) y la Parte 4 (EDA categórico).
- El dataset limpio se guardó en `data/processed/train_clean.csv`, disponible vía
  `src.load.load_train_clean()`.
